# 29. Policy Gradient (REINFORCE)
**Use case:** Insurance — Renewal retention offers: which discount for which policyholder

A personal-auto carrier loses many policyholders at renewal, and its blanket 5% loyalty discount is expensive for customers who would have stayed anyway. The retention team wants a policy that chooses 0%, 5% or 10% per customer to maximise retained premium net of discount cost. Data available: tenure, premium, claims history, competitor quote gap and multi-policy flag, plus past renewal outcomes. Success means higher retained premium than any fixed discount on a held-out book, no unfair skew across protected groups, and compliance with state rules on price optimization.

> 💡 A shop manager tries different window displays. Each week, if sales were good, she becomes a bit more likely to reuse that display; if poor, a bit less likely. Over months, the best displays dominate, but she still occasionally tries others. That is REINFORCE.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: REINFORCE on a 3-arm discount bandit
Use plain numpy REINFORCE to learn which of three renewal discounts earns the most expected retained premium for a single customer type.

| 5W + H | |
|---|---|
| **Who** | A pricing analyst new to policy-gradient methods. |
| **What** | One customer type, three discount levels with hidden renewal probabilities 70%, 82%, 86%, and a $1,200 premium. |
| **When** | Before adding customer features: to see how sampling and reward-scaled gradients steer a policy. |
| **Where** | A short numpy script, no deep-learning library. |
| **Why** | Shows that the best arm is not the one with the highest renewal rate but the one with the best net premium. |
| **How** | Sample a discount from softmax(theta), simulate renewal, update theta += lr x reward x (one_hot - probs), for 3,000 customers. |

**🧠 ML concepts:** Stochastic policy via softmax, Sampling actions for exploration, Log-probability gradient, Reward-weighted updates, Expected value vs renewal rate  
**🐍 Python concepts:** Parallel lists with zip, Module-level constants, np.random.choice with p=, Conditional expressions, Tuple membership test (step in (...))

In [ ]:
# REINFORCE on a 3-arm bandit: which renewal discount (0%, 5%, 10%) keeps the most premium? (Concept)
import numpy as np

np.random.seed(42)
# 🐍 PY: parallel lists - index 0, 1, 2 line up across all three
discounts = [0.00, 0.05, 0.10]
renew_prob = [0.70, 0.82, 0.86]  # hidden truth: chance the customer renews at each discount
PREMIUM = 1200.0                  # annual premium in dollars

# 🧠 ML: expected reward for each arm = P(renew) * premium * (1 - discount)
for d, p in zip(discounts, renew_prob):
    print(f"discount {d:4.0%}: true expected retained premium = ${p * PREMIUM * (1 - d):7.2f}")

# 🧠 ML: the policy = softmax over 3 preference numbers (theta). Start indifferent
theta = np.zeros(3)
lr = 0.05

def softmax(z):
    # 🐍 PY: subtracting the max is a numerical-safety trick; it does not change the result
    e = np.exp(z - z.max())
    return e / e.sum()

for step in range(1, 3001):
    probs = softmax(theta)
    # 🧠 ML: sample an action FROM the policy (this is how policy-gradient explores)
    a = np.random.choice(3, p=probs)
    renewed = np.random.rand() < renew_prob[a]
    # 🐍 PY: a conditional expression - value_if_true if condition else value_if_false
    reward = PREMIUM * (1 - discounts[a]) if renewed else 0.0
    # 🧠 ML: grad of log pi(a) for softmax = one_hot(a) - probs
    grad_log_pi = -probs
    grad_log_pi[a] += 1
    # 🧠 ML: REINFORCE update: theta += lr * reward * grad log pi(a); reward scaled to $1000s
    theta += lr * (reward / 1000) * grad_log_pi
    if step in (1, 10, 100, 500, 1000, 3000):
        print(f"step {step:4d}  probs(0%,5%,10%) = {np.round(softmax(theta), 3)}")

best = int(np.argmax(softmax(theta)))
print(f"\nLearned policy favours a {discounts[best]:.0%} discount.")

**Try this next:**
- Change renew_prob for 10% to 0.92: does the policy switch to 10%?
- Replace reward with reward - 900 (a crude baseline) and compare how quickly it converges.
- Lower lr to 0.01 and see how many steps are needed.

## BUILD: Feature-conditioned softmax policy in PyTorch
Train a PyTorch softmax policy that picks a discount per customer from tenure, premium, claims and competitor gap, and compare it with blanket discounts.

| 5W + H | |
|---|---|
| **Who** | A retention analytics team at a personal-lines carrier. |
| **What** | 4,000 synthetic training and 1,000 test policyholders with tenure_years, annual_premium, claims_last_3y and competitor_quote_gap. |
| **When** | Before a pilot, using a response simulator calibrated from past renewal campaigns. |
| **Where** | A PyTorch script; the policy is a single linear layer producing three logits. |
| **Why** | New customers and those with a cheaper rival quote respond to discounts; loyal customers do not. A personalised policy avoids wasting discount on the loyal. |
| **How** | Mini-batches of 256 customers, sample discounts, simulate renewals, minimise -(log pi x reward), then compare expected value on the test book. |

**🧠 ML concepts:** Contextual policy (features -> action probabilities), Mini-batch REINFORCE, Simulated environment for offline learning, Comparing to fixed strategies and an oracle, Interpreting policy behaviour by segment  
**🐍 Python concepts:** DataFrame from a dict of arrays, np.c_ to stack columns, Vectorised pandas arithmetic and clip, torch.distributions.Categorical, pd.cut for banding, groupby with observed=True

In [ ]:
# REINFORCE with a feature-conditioned softmax policy: personalised renewal discounts (Build)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
DISCOUNTS = np.array([0.00, 0.05, 0.10])


def make_customers(n):
    # 🐍 PY: a DataFrame from a dict of equal-length numpy arrays
    return pd.DataFrame({
        "tenure_years": rng.integers(1, 15, n),
        "annual_premium": rng.normal(1400, 400, n).clip(400, 3000).round(0),
        "claims_last_3y": rng.poisson(0.4, n),
        "competitor_quote_gap": rng.normal(0.0, 0.08, n).round(3),  # + means rival is cheaper
    })


def renew_probability(df, discount):
    # 🧠 ML: the hidden 'world': loyal customers barely react to discounts, shoppers react a lot
    z = (0.6 + 0.12 * df.tenure_years - 0.3 * df.claims_last_3y - 12 * df.competitor_quote_gap
         + discount * (30 - 3 * df.tenure_years).clip(lower=1))
    return 1 / (1 + np.exp(-z))


def features(df):
    # 🧠 ML: scale features to similar ranges so gradient steps treat them fairly
    X = np.c_[df.tenure_years / 10, df.annual_premium / 1000, df.claims_last_3y, df.competitor_quote_gap * 10]
    return torch.tensor(X, dtype=torch.float32)


train, test = make_customers(4000), make_customers(1000)
X_train, X_test = features(train), features(test)
# 🧠 ML: policy network = linear layer -> 3 logits -> softmax probabilities over the discount levels
policy = nn.Linear(4, 3)
opt = torch.optim.Adam(policy.parameters(), lr=0.03)

for epoch in range(1, 401):
    idx = rng.choice(len(train), 256, replace=False)
    dist = torch.distributions.Categorical(logits=policy(X_train[idx]))
    a = dist.sample()
    batch = train.iloc[idx]
    d = DISCOUNTS[a.numpy()]
    # 🧠 ML: simulate the customer's choice; reward = retained premium net of the discount
    renewed = rng.random(len(idx)) < renew_probability(batch, d).to_numpy()
    reward = torch.tensor(renewed * batch.annual_premium.to_numpy() * (1 - d) / 1000, dtype=torch.float32)
    # 🧠 ML: REINFORCE loss = -(log pi(a|x) * reward); minimising it raises probability of rewarded actions
    loss = -(dist.log_prob(a) * reward).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  avg reward ${reward.mean().item() * 1000:7.1f}  "
              f"action mix (0/5/10%) {np.round(np.bincount(a.numpy(), minlength=3) / len(idx), 2)}")


def expected_value(df, d):
    # 🐍 PY: vectorised pandas maths - no loop over customers needed
    return (renew_probability(df, d) * df.annual_premium * (1 - d)).mean()


with torch.no_grad():
    choice = policy(X_test).argmax(1).numpy()
print("\nTest set: expected retained premium per customer")
for i, dsc in enumerate(DISCOUNTS):
    print(f"  everyone gets {dsc:4.0%}:          ${expected_value(test, np.full(len(test), dsc)):8.2f}")
print(f"  learned personalised policy:  ${expected_value(test, DISCOUNTS[choice]):8.2f}")
# 🧠 ML: the 'oracle' knows the hidden world - an upper bound we can't reach in real life
oracle = np.max([renew_probability(test, dd) * test.annual_premium * (1 - dd) for dd in DISCOUNTS], axis=0)
print(f"  oracle (perfect knowledge):   ${oracle.mean():8.2f}")

test["offer"] = DISCOUNTS[choice]
# 🐍 PY: pd.cut buckets tenure into bands; groupby + mean summarises the offer per band
test["tenure_band"] = pd.cut(test.tenure_years, [0, 3, 7, 15], labels=["1-3y", "4-7y", "8-14y"])
print("\nAverage discount offered by tenure band:")
print(test.groupby("tenure_band", observed=True)["offer"].mean().map("{:.1%}".format).to_string())

**Try this next:**
- Increase training to 1,000 epochs: does the policy get closer to the oracle?
- Replace nn.Linear(4, 3) with a small two-layer network and compare.
- Subtract the batch-average reward from reward before the loss and see whether learning is smoother.

## PRODUCTION: REINFORCE with baseline, entropy bonus and fairness audit
Build a production-style renewal-offer policy with a learned baseline and entropy bonus, choose settings on a validation book, beat fixed discounts on a test book and audit fairness.

| 5W + H | |
|---|---|
| **Who** | A pricing and retention team preparing a filing-ready proposal for actuarial, compliance and model risk. |
| **What** | 6,000 train, 2,000 validation and 2,000 test policyholders with five features; age band kept aside for auditing only. |
| **When** | Before a controlled pilot in states where personalised retention discounts are permitted. |
| **Where** | An offline training job; the policy and its scaling parameters are saved with torch.save. |
| **Why** | Plain REINFORCE is noisy. A baseline cuts variance, entropy prevents early lock-in, and regulators expect fairness evidence and documented limits. |
| **How** | Ablation of baseline and entropy, selection on validation, test-book comparison, partial-dependence-style table, age-band audit, regulatory note. |

**🧠 ML concepts:** Baseline subtraction for variance reduction, Entropy regularisation, Ablation study, Validation-based model selection, Explainability through segment tables, Fairness auditing on attributes not used by the model  
**🐍 Python concepts:** Default None arguments to reuse training statistics, nn.Module with two sub-networks, retain_graph for two backward passes, Generator expressions inside sum(), Format conversion !s in f-strings, pivot_table and DataFrame.map

In [ ]:
# REINFORCE with learned baseline + entropy bonus for renewal discounts, with fairness checks (Production)
import os
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_num_threads(1)
SEED = 42
DISCOUNTS = np.array([0.00, 0.05, 0.10])
FEATS = ["tenure_years", "annual_premium", "claims_last_3y", "competitor_quote_gap", "multi_policy"]


def make_book(n, rng):
    df = pd.DataFrame({
        "tenure_years": rng.integers(1, 15, n),
        "annual_premium": rng.normal(1400, 400, n).clip(400, 3000).round(0),
        "claims_last_3y": rng.poisson(0.4, n),
        "competitor_quote_gap": rng.normal(0.0, 0.08, n),
        "multi_policy": rng.random(n) < 0.35,
        # 🧠 ML: age_band is NOT a model input - we keep it only to AUDIT outcomes for fairness
        "age_band": rng.choice(["18-34", "35-54", "55+"], n, p=[0.3, 0.4, 0.3]),
    })
    return df


def renew_prob(df, d):
    # 🧠 ML: hidden response curve - shoppers (short tenure, rival cheaper) are discount-sensitive
    sens = (30 - 3 * df.tenure_years).clip(lower=1) + 40 * df.competitor_quote_gap.clip(lower=0)
    z = (0.6 + 0.12 * df.tenure_years - 0.3 * df.claims_last_3y - 12 * df.competitor_quote_gap
         + 0.8 * df.multi_policy + d * sens)
    return 1 / (1 + np.exp(-z))


def to_tensor(df, mu=None, sd=None):
    X = df[FEATS].astype(float).to_numpy()
    # 🐍 PY: default None args let us reuse TRAIN mean/std on the test set (no leakage)
    mu = X.mean(0) if mu is None else mu
    sd = X.std(0) if sd is None else sd
    return torch.tensor((X - mu) / sd, dtype=torch.float32), mu, sd


class Policy(nn.Module):
    def __init__(self, n_in, n_act=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, n_act))
        # 🧠 ML: separate value head = learned baseline b(x): 'what reward do we expect for this customer?'
        self.base = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x), self.base(x).squeeze(-1)


def train(df, X, use_baseline, ent_coef, epochs=300, batch=256, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pol = Policy(X.shape[1])
    opt = torch.optim.Adam(pol.parameters(), lr=0.01)
    grad_norms = []
    for _ in range(epochs):
        idx = rng.choice(len(df), batch, replace=False)
        logits, b = pol(X[idx])
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        sub, d = df.iloc[idx], DISCOUNTS[a.numpy()]
        renewed = rng.random(batch) < renew_prob(sub, d).to_numpy()
        r = torch.tensor(renewed * sub.annual_premium.to_numpy() * (1 - d) / 1000, dtype=torch.float32)
        # 🧠 ML: advantage = reward - baseline. Same expected gradient, MUCH lower variance
        adv = r - b.detach() if use_baseline else r
        pg_loss = -(dist.log_prob(a) * adv).mean()
        # 🧠 ML: entropy bonus keeps the policy from collapsing onto one discount too early
        loss = pg_loss - ent_coef * dist.entropy().mean() + (b - r).pow(2).mean()
        opt.zero_grad()
        pg_loss.backward(retain_graph=True)
        # 🐍 PY: generator expression inside sum() - total gradient size of the policy layer
        grad_norms.append(sum(p.grad.norm().item() ** 2 for p in pol.net.parameters()) ** 0.5)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return pol, np.std(grad_norms[-100:])


def expected_premium(df, d):
    return float((renew_prob(df, d) * df.annual_premium * (1 - d)).mean())


def main():
    rng = np.random.default_rng(SEED)
    train_df, valid_df, test_df = make_book(6000, rng), make_book(2000, rng), make_book(2000, rng)
    X_tr, mu, sd = to_tensor(train_df)
    X_va, _, _ = to_tensor(valid_df, mu, sd)
    X_te, _, _ = to_tensor(test_df, mu, sd)

    # 🧠 ML: ablation + tuning - does the baseline help, and how much entropy bonus is right?
    print("config                         grad-norm std  valid $/customer")
    results = {}
    for use_b, ent in [(False, 0.0), (True, 0.0), (True, 0.01), (True, 0.05)]:
        pol, gstd = train(train_df, X_tr, use_b, ent)
        with torch.no_grad():
            val = expected_premium(valid_df, DISCOUNTS[pol(X_va)[0].argmax(1).numpy()])
        key = f"baseline={use_b!s:5s} entropy={ent:.2f}"
        results[key] = (pol, val)
        print(f"{key:30s} {gstd:13.3f} {val:16.2f}")
    # 🧠 ML: pick the config on the VALIDATION book; the test book is scored once, below
    best_key = max(results, key=lambda k: results[k][1])
    pol = results[best_key][0]
    with torch.no_grad():
        choice = pol(X_te)[0].argmax(1).numpy()
    best_val = expected_premium(test_df, DISCOUNTS[choice])
    print(f"Chosen: {best_key}")

    print("\nStrategy comparison on held-out book (expected retained premium net of discount):")
    for d in DISCOUNTS:
        print(f"  fixed {d:4.0%} for everyone   ${expected_premium(test_df, np.full(len(test_df), d)):8.2f}")
    print(f"  REINFORCE policy         ${best_val:8.2f}")
    fixed_best = max(expected_premium(test_df, np.full(len(test_df), d)) for d in DISCOUNTS)
    print(f"  uplift vs best fixed: ${best_val - fixed_best:.2f}/customer "
          f"= ${(best_val - fixed_best) * 500_000 / 1e6:.2f}M on a 500k-policy book")

    # 🧠 ML: explainability - average offer by driver (a simple partial-dependence-style table)
    test_df["offer"] = DISCOUNTS[choice]
    test_df["tenure_band"] = pd.cut(test_df.tenure_years, [0, 3, 7, 15], labels=["1-3y", "4-7y", "8-14y"])
    test_df["rival_cheaper"] = test_df.competitor_quote_gap > 0.05
    print("\nAverage offer by tenure band and whether a rival quote is >5% cheaper:")
    print(test_df.pivot_table(index="tenure_band", columns="rival_cheaper", values="offer",
                              observed=True).map(lambda v: f"{v:.1%}").to_string())

    # 🧠 ML: fairness audit - are outcomes skewed by age, even though age was never an input?
    test_df["p_renew"] = renew_prob(test_df, test_df.offer)
    audit = test_df.groupby("age_band").agg(avg_offer=("offer", "mean"), avg_premium=("annual_premium", "mean"),
                                            renew_rate=("p_renew", "mean"))
    print("\nFairness audit by age band (not a model input):")
    print(audit.round(3).to_string())

    path = os.path.join(tempfile.mkdtemp(), "renewal_policy.pt")
    torch.save({"state_dict": pol.state_dict(), "mu": mu, "sd": sd}, path)
    print(f"\nSaved policy to {path}")
    print("REGULATORY NOTE: many US states restrict or ban 'price optimization' - setting renewal prices")
    print("by predicted willingness to stay rather than risk (see NAIC 2015 white paper, CA/NY/FL bulletins).")
    print("Where banned, use this only for non-price retention actions; elsewhere, file rating rules, keep")
    print("discounts within approved bands, and test for disparate impact before deployment.")


if __name__ == "__main__":
    main()

**Try this next:**
- Add age_band to FEATS (one-hot) and rerun the audit: what changes, and why would compliance object?
- Set entropy to 0.2: does the policy become too random to beat fixed 5%?
- Make the competitor gap twice as volatile and check whether the uplift grows or shrinks.